Name : Diksha Parulekar.
Roll no. : 23102A0068.
Subject : R programming assignment 3

In [1]:
install.packages(c("dplyr","readr","jsonlite","readxl","writexl","DBI","RSQLite","lubridate"), quiet = TRUE)

library(dplyr)
library(readr)
library(jsonlite)
library(readxl)
library(writexl)
library(DBI)
library(RSQLite)
library(lubridate)

cat("Packages loaded successfully\n")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




Packages loaded successfully


In [2]:
set.seed(42)

# Try downloading the UCI Online Retail dataset
uci_url <- "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
temp_file <- "OnlineRetail_raw.xlsx"

download_ok <- tryCatch({
  download.file(uci_url, temp_file, mode = "wb", quiet = TRUE)
  TRUE
}, error = function(e) FALSE)

if (download_ok && file.exists(temp_file) && file.size(temp_file) > 1000) {
  raw_data <- read_excel(temp_file)
  cat("UCI dataset downloaded successfully. Rows:", nrow(raw_data), "\n")
} else {
  cat("UCI download failed — generating synthetic Online Retail-style data\n")
  n <- 3000
  countries <- c("United Kingdom","Germany","France","EIRE","Spain","Netherlands","Belgium","Australia")
  raw_data <- tibble(
    InvoiceNo = sample(10000:10500, n, replace = TRUE),
    StockCode = paste0("SC", sample(100:199, n, replace = TRUE)),
    Description = paste("Product", sample(100:199, n, replace = TRUE)),
    Quantity = sample(c(-5, 0, 1:50), n, replace = TRUE, prob = c(0.02,0.02, rep(0.96/50,50))),
    InvoiceDate = as.character(sample(seq(as.Date('2011-01-01'), as.Date('2011-12-31'), by="day"), n, replace = TRUE)),
    UnitPrice = round(sample(c(0, runif(n, 0.5, 100)), n, replace = TRUE), 2),
    CustomerID = sample(c(NA, 12000:12200), n, replace = TRUE, prob = c(0.05, rep(0.95/201,201))),
    Country = sample(countries, n, replace = TRUE)
  )
}

head(raw_data)

UCI dataset downloaded successfully. Rows: 541909 


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
<chr>,<chr>,<chr>,<dbl>,<dttm>,<dbl>,<dbl>,<chr>
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom


In [3]:
# transactions.csv
transactions_raw <- raw_data %>%
  select(InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate) %>%
  distinct()
write_csv(transactions_raw, "transactions.csv")

# products.json
products_raw <- raw_data %>%
  select(StockCode, Description, UnitPrice) %>%
  distinct(StockCode, .keep_all = TRUE)
write_json(products_raw, "products.json")

# customers.xlsx
customers_raw <- raw_data %>%
  filter(!is.na(CustomerID)) %>%
  select(CustomerID, Country) %>%
  distinct(CustomerID, .keep_all = TRUE)
write_xlsx(customers_raw, "customers.xlsx")

cat("Source files created:\n")
cat("transactions.csv:", nrow(transactions_raw), "rows\n")
cat("products.json:", nrow(products_raw), "rows\n")
cat("customers.xlsx:", nrow(customers_raw), "rows\n")

Source files created:
transactions.csv: 536480 rows
products.json: 4070 rows
customers.xlsx: 4372 rows


Task 1 — Import and Clean the Data


In [4]:
# Import from the three sources
transactions <- read_csv("transactions.csv", show_col_types = FALSE)
products <- fromJSON("products.json") %>% as_tibble()
customers <- read_excel("customers.xlsx")

cat("Raw counts -> Transactions:", nrow(transactions),
    "| Products:", nrow(products),
    "| Customers:", nrow(customers), "\n")

# --- Cleaning: Transactions ---
transactions_clean <- transactions %>%
  filter(!is.na(CustomerID), !is.na(InvoiceNo), !is.na(StockCode)) %>%   # remove missing keys
  distinct() %>%                                                         # remove duplicates
  filter(Quantity > 0)                                                   # remove invalid/zero quantity

# --- Cleaning: Products ---
products_clean <- products %>%
  filter(!is.na(UnitPrice), UnitPrice > 0) %>%                           # remove invalid/zero price
  distinct(StockCode, .keep_all = TRUE)

# --- Cleaning: Customers ---
customers_clean <- customers %>%
  filter(!is.na(CustomerID), !is.na(Country)) %>%
  distinct(CustomerID, .keep_all = TRUE)

cat("\nCleaned counts -> Transactions:", nrow(transactions_clean),
    "| Products:", nrow(products_clean),
    "| Customers:", nrow(customers_clean), "\n")

cat("\nCleaning decisions:\n")
cat("- Removed rows with missing CustomerID/InvoiceNo/StockCode (can't be linked)\n")
cat("- Removed exact duplicate transaction rows\n")
cat("- Removed transactions with Quantity <= 0 (returns/invalid entries)\n")
cat("- Removed products with UnitPrice <= 0 or missing (data entry errors)\n")

Raw counts -> Transactions: 536480 | Products: 4070 | Customers: 4372 

Cleaned counts -> Transactions: 392708 | Products: 3855 | Customers: 4372 

Cleaning decisions:
- Removed rows with missing CustomerID/InvoiceNo/StockCode (can't be linked)
- Removed exact duplicate transaction rows
- Removed transactions with Quantity <= 0 (returns/invalid entries)
- Removed products with UnitPrice <= 0 or missing (data entry errors)


Task 2 — Integrate the Multiple Data Sources


In [5]:
# Join transactions -> products (inner_join: only keep transactions with valid product info)
integrated <- transactions_clean %>%
  inner_join(products_clean, by = "StockCode")

# Join with customers (left_join: keep all transactions even if some customer info missing)
integrated <- integrated %>%
  left_join(customers_clean, by = "CustomerID")

# Add Revenue column
integrated <- integrated %>%
  mutate(Revenue = Quantity * UnitPrice)

cat("Final integrated dataset dimensions:", dim(integrated)[1], "rows x", dim(integrated)[2], "cols\n")

# Unmatched records check
unmatched_country <- integrated %>% filter(is.na(Country)) %>% nrow()
cat("Records with unmatched customer/country info:", unmatched_country, "\n")

cat("\nJoin justification:\n")
cat("- inner_join() used for products: a transaction is meaningless without valid product/price data\n")
cat("- left_join() used for customers: we retain all valid sales transactions even if\n")
cat("  a customer record is missing, to avoid losing revenue data\n")

head(integrated)

Final integrated dataset dimensions: 387877 rows x 9 cols
Records with unmatched customer/country info: 0 

Join justification:
- inner_join() used for products: a transaction is meaningless without valid product/price data
- left_join() used for customers: we retain all valid sales transactions even if
  a customer record is missing, to avoid losing revenue data


InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,Description,UnitPrice,Country,Revenue
<chr>,<chr>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<chr>,<dbl>
536365,85123A,17850,6,2010-12-01 08:26:00,WHITE HANGING HEART T-LIGHT HOLDER,2.55,United Kingdom,15.30
536365,71053,17850,6,2010-12-01 08:26:00,WHITE METAL LANTERN,3.39,United Kingdom,20.34
536365,84406B,17850,8,2010-12-01 08:26:00,CREAM CUPID HEARTS COAT HANGER,2.75,United Kingdom,22.00
536365,84029G,17850,6,2010-12-01 08:26:00,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,United Kingdom,20.34
536365,84029E,17850,6,2010-12-01 08:26:00,RED WOOLLY HOTTIE WHITE HEART.,3.39,United Kingdom,20.34
536365,22752,17850,2,2010-12-01 08:26:00,SET 7 BABUSHKA NESTING BOXES,7.65,United Kingdom,15.30


Task 3 — Sales and Customer Analysis

In [6]:
# 1. Total sales revenue
total_revenue <- sum(integrated$Revenue, na.rm = TRUE)
cat("Total Sales Revenue:", round(total_revenue, 2), "\n\n")

# 2. Top 5 products by revenue
top5_products <- integrated %>%
  group_by(StockCode, Description) %>%
  summarise(TotalRevenue = sum(Revenue, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(TotalRevenue)) %>%
  head(5)
cat("Top 5 Products by Revenue:\n"); print(top5_products)

# 3. Top 5 countries by revenue
top5_countries <- integrated %>%
  filter(!is.na(Country)) %>%
  group_by(Country) %>%
  summarise(TotalRevenue = sum(Revenue, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(TotalRevenue)) %>%
  head(5)
cat("\nTop 5 Countries by Revenue:\n"); print(top5_countries)

# 4. Top 5 customers by purchase value
top5_customers <- integrated %>%
  filter(!is.na(CustomerID)) %>%
  group_by(CustomerID) %>%
  summarise(TotalRevenue = sum(Revenue, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(TotalRevenue)) %>%
  head(5)
cat("\nTop 5 Customers by Revenue:\n"); print(top5_customers)

# Customer value classification using case_when()
customer_value <- integrated %>%
  filter(!is.na(CustomerID)) %>%
  group_by(CustomerID) %>%
  summarise(CustomerRevenue = sum(Revenue, na.rm = TRUE), .groups = "drop") %>%
  mutate(ValueSegment = case_when(
    CustomerRevenue < 100 ~ "Low Value",
    CustomerRevenue >= 100 & CustomerRevenue < 500 ~ "Medium Value",
    CustomerRevenue >= 500 & CustomerRevenue < 1000 ~ "High Value",
    CustomerRevenue >= 1000 ~ "Premium"
  ))

cat("\nCustomer Segment Distribution:\n")
print(table(customer_value$ValueSegment))

# High-performing vs underperforming market
best_market <- top5_countries$Country[1]
worst_market <- integrated %>%
  filter(!is.na(Country)) %>%
  group_by(Country) %>%
  summarise(TotalRevenue = sum(Revenue, na.rm = TRUE), .groups = "drop") %>%
  arrange(TotalRevenue) %>%
  head(1)

cat("\nHigh-performing market:", best_market,
    "- highest total revenue, indicating strong customer base and demand\n")
cat("Underperforming market:", worst_market$Country,
    "- lowest revenue (", round(worst_market$TotalRevenue,2),
    "), suggesting limited reach or low purchase frequency\n")

Total Sales Revenue: 10752840 

Top 5 Products by Revenue:
# A tibble: 5 × 3
  StockCode Description                        TotalRevenue
  <chr>     <chr>                                     <dbl>
1 23843     PAPER CRAFT , LITTLE BIRDIE             168470.
2 47566     PARTY BUNTING                           142438.
3 22423     REGENCY CAKESTAND 3 TIER                135605.
4 85123A    WHITE HANGING HEART T-LIGHT HOLDER       93746.
5 23166     MEDIUM CERAMIC TOP STORAGE JAR           81033.

Top 5 Countries by Revenue:
# A tibble: 5 × 2
  Country        TotalRevenue
  <chr>                 <dbl>
1 United Kingdom     8861857.
2 Netherlands         363884.
3 EIRE                331660.
4 Germany             263819.
5 France              226976.

Top 5 Customers by Revenue:
# A tibble: 5 × 2
  CustomerID TotalRevenue
       <dbl>        <dbl>
1      18102      408760.
2      14646      357531.
3      17450      186038.
4      14911      182691.
5      16446      168472.

Customer Segment

Task 4 — Store and Retrieve Data Using SQL

In [7]:
# Create SQLite database
con <- dbConnect(RSQLite::SQLite(), "retail_sales.db")

# Export cleaned/integrated dataset
dbWriteTable(con, "retail_sales", integrated, overwrite = TRUE)
cat("Table 'retail_sales' created with", dbGetQuery(con, "SELECT COUNT(*) as n FROM retail_sales")$n, "rows\n\n")

# Query 1: Top 5 customers by revenue
query1 <- dbGetQuery(con, "
  SELECT CustomerID, SUM(Revenue) AS TotalRevenue
  FROM retail_sales
  WHERE CustomerID IS NOT NULL
  GROUP BY CustomerID
  ORDER BY TotalRevenue DESC
  LIMIT 5
")
cat("SQL Query 1 - Top 5 Customers by Revenue:\n")
print(query1)

# Query 2: Total revenue by country
query2 <- dbGetQuery(con, "
  SELECT Country, SUM(Revenue) AS TotalRevenue
  FROM retail_sales
  WHERE Country IS NOT NULL
  GROUP BY Country
  ORDER BY TotalRevenue DESC
")
cat("\nSQL Query 2 - Total Revenue by Country:\n")
print(query2)

dbDisconnect(con)
cat("\nDatabase connection closed. retail_sales.db saved.\n")

Table 'retail_sales' created with 387877 rows

SQL Query 1 - Top 5 Customers by Revenue:
  CustomerID TotalRevenue
1      18102     408760.0
2      14646     357531.1
3      17450     186038.0
4      14911     182690.5
5      16446     168472.5

SQL Query 2 - Total Revenue by Country:
                Country TotalRevenue
1        United Kingdom   8861857.13
2           Netherlands    363884.48
3                  EIRE    331660.17
4               Germany    263818.97
5                France    226975.60
6             Australia    173918.61
7                 Spain     67426.09
8           Switzerland     66619.97
9                 Japan     48600.22
10              Belgium     47858.02
11               Sweden     43652.09
12               Norway     40283.20
13             Portugal     32679.45
14              Finland     23306.79
15      Channel Islands     22059.72
16              Denmark     21291.79
17                Italy     21065.45
18               Cyprus     16829.62
19         

In [8]:
cat("=== BUSINESS INSIGHTS ===\n\n")
cat("1. Revenue is highly concentrated in a few top products and countries,\n")
cat("   suggesting the business should prioritize inventory and marketing spend\n")
cat("   around these high-performing categories and regions.\n\n")
cat("2. A significant share of customers fall into the 'Low Value' segment,\n")
cat("   indicating an opportunity for targeted upselling or loyalty programs\n")
cat("   to move them toward 'Medium' and 'High Value' tiers.\n\n")
cat("3. The gap between the top-performing and underperforming markets highlights\n")
cat("   an expansion opportunity — underperforming regions may benefit from\n")
cat("   localized promotions or better product-market fit analysis.\n")

# Save cleaned integrated dataset for submission
write_csv(integrated, "final_integrated_retail_data.csv")
cat("\nFinal dataset exported as final_integrated_retail_data.csv\n")
cat("Files ready for submission: transactions.csv, products.json, customers.xlsx,\n")
cat("final_integrated_retail_data.csv, retail_sales.db\n")

=== BUSINESS INSIGHTS ===

1. Revenue is highly concentrated in a few top products and countries,
   suggesting the business should prioritize inventory and marketing spend
   around these high-performing categories and regions.

2. A significant share of customers fall into the 'Low Value' segment,
   indicating an opportunity for targeted upselling or loyalty programs
   to move them toward 'Medium' and 'High Value' tiers.

3. The gap between the top-performing and underperforming markets highlights
   an expansion opportunity — underperforming regions may benefit from
   localized promotions or better product-market fit analysis.

Final dataset exported as final_integrated_retail_data.csv
Files ready for submission: transactions.csv, products.json, customers.xlsx,
final_integrated_retail_data.csv, retail_sales.db
